# Phase 5 — a group's past, and the horizon it goes stale across

Every per-group feature the harness could build averaged the group's rows
together: the mean target (`fit_encoding`), the mean/median/spread of some other
column (`fit_group_stats`). All of them throw away the **order** the rows came
in. Nothing in the vocabulary could say *what this store sold most recently*,
which is a statement about **when**, not about **how much**.

This notebook builds that, by hand first. The hard part is not storing the past —
it is deciding what the past means for a row the model has never seen, sitting
some unknown distance after everything you know. Two decision records answer that:

| record | question | answer |
|---|---|---|
| **0047** | what does a lag serve past the end of stored history? | the last known value **plus its age**, and age is per row |
| **0048** | may a validation row read its own neighbours' labels? | no — history is frozen per fold at the training prefix |

Section 6 is the one that matters. Everything before it is machinery.

## 1. The data: a panel where recency actually matters

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)


def make_panel(n_days=120):
    """A store-day panel whose sales *wander* rather than sit at a level.

    Each store's sales follow a random walk. That matters for this notebook:
    if sales were a fixed per-store level plus noise, `fit_group_stats` would
    already capture everything and there would be nothing for history to add.
    Here, *when* you last saw a store tells you more than what it usually sells.
    """
    level = {"s1": 30.0, "s2": 34.0, "s3": 38.0, "s4": 42.0}
    start = pd.Timestamp("2015-01-01")
    rows = []
    for store, base in level.items():
        walk = base + np.cumsum(rng.normal(scale=1.5, size=n_days))
        for i, d in enumerate(pd.date_range(start, periods=n_days)):
            rows.append({"store": store, "date": d,
                         "promo": int(rng.random() > 0.7),
                         "sales": walk[i]})
    return pd.DataFrame(rows).sort_values(["date", "store"]).reset_index(drop=True)


panel = make_panel()
panel.head()

## 2. Why this needed a tool

The obvious way to write "yesterday's value" is one line of pandas. Try it in the
free-code path.

In [ ]:
from tabular_agent.transform import transform_features

CODE = """
def transform_features(df):
    df = df.copy()
    df["promo_yesterday"] = df.groupby("store")["promo"].shift(1)
    return df
"""

try:
    transform_features(panel, CODE)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:300]}")

That is 0011's stability check, and it is **right**. Only the *code* is stored, so
at predict time it re-runs `shift(1)` over whatever rows arrived in that request.
One row alone gets `NaN`; the same row batched with its neighbours gets something
else. The feature depends on who it was batched with, which is not a property a
feature may have.

The same refusal fires for the target, on top of the target being invisible to
transforms at all (0009):

In [ ]:
CODE_TARGET = """
def transform_features(df):
    df = df.copy()
    df["sales_yesterday"] = df.groupby("store")["sales"].shift(1)
    return df
"""

try:
    transform_features(panel, CODE_TARGET)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:300]}")

So the capability was blocked from both directions, and correctly from both. What
makes it legal is not different arithmetic — it is that the number comes from
**stored state** rather than from the frame in front of it. That is the same thing
that makes `fit_group_stats` legal where `groupby().transform("mean")` is not.

## 3. `HistoryEncoder` by hand

Fit it on the whole panel and look at what it kept.

In [ ]:
from tabular_agent.encoders import HistoryEncoder

enc = HistoryEncoder(["store"], "sales", "date").fit(panel)

print("history ends at:", pd.Timestamp("1970-01-01") + pd.Timedelta(days=enc.frozen_at_))
print("columns:        ", enc.output_columns)
print("stored per group:", {k: len(v[0]) for k, v in enc.history_.items()})

One observation per store, because nothing asked for more. Now serve it rows from
after the end of history — the real case, where the model is asked about days it
has never seen.

In [ ]:
future = pd.DataFrame({
    "store": ["s1"] * 4,
    "date": pd.to_datetime(["2015-05-01", "2015-05-05", "2015-05-15", "2015-07-01"]),
})

# Indexed by date so the frame being rounded is all numbers.
enc.transform(future).set_index("date").round(2)

**The value never changes. The age does.** That pair is the whole design (0047).

Serving the value alone would be a carry-forward: the same number, drifting
further from the truth the further ahead you predict, and saying nothing about
it. This project's characteristic failure is a number that looks fine and
describes something else, and a bare carry-forward is that failure in feature
form. The age column is what lets a model discount a value it should not trust.

And because the answer comes from storage, it does not depend on the batch:

In [ ]:
alone = enc.transform(future.iloc[[3]])
batch = enc.transform(future)

print("alone:", round(alone["store_sales_latest"].iloc[0], 2), "age", alone["store_sales_latest_ago"].iloc[0])
print("batch:", round(batch["store_sales_latest"].iloc[3], 2), "age", batch["store_sales_latest_ago"].iloc[3])

## 4. Storage is bounded, on purpose

An encoder lives in a table's lineage and is carried in every bundle. If its size
grew with the length of the table, it could not. `keep` is a per-group cap, so the
encoder is (number of groups × `keep`) and never a function of how many rows
there are.

In [ ]:
big = make_panel(n_days=2000)

for keep in (1, 8):
    e = HistoryEncoder(["store"], "sales", "date", keep=keep).fit(big)
    stored = sum(len(v[0]) for v in e.history_.values())
    print(f"keep={keep}: {stored:>3} observations stored, for {len(big)} rows of input")

## 5. Real lags, and the horizon that makes them legal

`_latest` needs no declaration to be honest. A *lag* does. "The value 7 days ago" is
only answerable from stored history if every row you are asked about is at least 7
days past the end of it — so the task has to say how far ahead it forecasts.

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.encoders import fit_history

folds = make_folds(panel["sales"], "regression", n_splits=4,
                   order=panel["date"].to_numpy())

try:
    fit_history(panel, ["store"], "sales", "date", folds=folds, offsets=[1])
except ValueError as e:
    print("REFUSED:", e)

`TaskSpec.forecast_horizon` is where that declaration goes — a fact about the
serving contract, which makes it the data owner's to state (0029). It is a
**minimum**, not a constant: within one predict batch the rows sit at different
distances from the end of history, so it cannot be a single number applied to
every row. Per-row distance is what `_latest_ago` reports. The horizon's only job is
to say which offsets are **legal**.

**Legal is not the same as useful**, and that distinction cost Rossmann v8 a
table (0050). An offset shorter than the *whole period you predict* is answerable
for the earliest rows and, for the rest, quietly falls back to the last-known
value. The declared floor is the gap to the *first* row; what an offset actually
needs to clear is the *span of the block*. `fit_history` reports
`offsets_answerable` for exactly this reason.


In [ ]:
try:
    fit_history(panel, ["store"], "sales", "date", folds=folds,
                offsets=[1], forecast_horizon=7)
except ValueError as e:
    print("REFUSED:", e)

At or beyond the declared horizon, an offset is legal. And when it cannot be
answered exactly — a row far past the end of history — it degrades into `_latest`
and **reports the age it actually found**, rather than a stale number under a name
that promises a fresh one:

In [ ]:
enc7 = HistoryEncoder(["store"], "sales", "date", offsets=[7], keep=10).fit(panel)

out = enc7.transform(future)
out.set_index("date")[["store_sales_7_ago", "store_sales_7_ago_exact"]].round(2)

The first two rows get a genuine 7-day lag, so `_7_ago_exact` is 1. The last two
are far enough out that "7 days ago" is past the end of history, so they get the
newest thing available and `_exact` drops to **0** — the column is saying it has
become `_latest` under a name that promises a fixed lag.

A flag rather than an age here, because for an offset the age is near-constant —
it equals the offset whenever the lookup worked — so all of its information is in
the failures. `_latest` keeps a real age, which genuinely varies row to row.

## 6. The per-fold freeze — the part that matters (0048)

Everything so far was about *predict* time. Now: what may a **validation** row
read?

`docs/time_aware_features.md` §3.4 framed this as a leakage question and reached
for out-of-fold construction, the guard `fit_group_stats` uses. That framing is
wrong, and 0048 exists to say why. Two different failures wear the same word:

1. **Self-inclusion** — the row's own target inside its own feature. This is what
   out-of-fold removes. A strictly backward lag **cannot commit it**: the row at
   `t` only ever reads `t-k`. Out-of-fold buys nothing here.
2. **Availability skew** — reading a label that will not exist when it is served.
   Real, severe, and *not* what out-of-fold fixes.

So the guard a target lag needs is the **horizon, not the fold**. The mechanism
still ends up fold-shaped — each fold's history is frozen at the end of its
training prefix — but for the second reason, which changes what a correct
implementation looks like.

In [ ]:
oof, final = fit_history(panel, ["store"], "sales", "date", folds=folds)

frozen = oof["store_sales_latest"]
frozen_age = oof["store_sales_latest_ago"]

view = pd.concat([panel[["store", "date", "sales"]], oof], axis=1)
view[view["store"] == "s1"].iloc[22:32].set_index("date").round(2)

Rows in the first block keep `NaN` — under forward chaining nothing precedes them
to train on, the same gap out-of-fold target encoding leaves.

Then look at what happens *inside* a block: the value stands still and the age
climbs. The feature goes stale across the validation block exactly as it would go
stale across a real forecast horizon.

In [ ]:
_, va = folds[-1]
s1_rows = va[(panel.iloc[va]["store"] == "s1").to_numpy()]

print("age across the last validation block:")
print(frozen_age.iloc[s1_rows].dropna().astype(int).tolist())

Compare that with the naive backward lag — the one that reads whatever row
happens to sit before it, regardless of fold.

In [ ]:
naive = panel.groupby("store")["sales"].shift(1)
seen = frozen.notna() & naive.notna()

# Compared *within* store: the between-store level differences would otherwise
# make any store-flavoured number look near-perfect and hide the whole effect.
dev = panel["sales"] - panel.groupby("store")["sales"].transform("mean")


def within_store_corr(feature):
    f = feature - feature.groupby(panel["store"]).transform("mean")
    return round(float(np.corrcoef(f[seen], dev[seen])[0, 1]), 3)


print(f"naive lag-1 : age always 1, corr with y = {within_store_corr(naive)}")
print(f"frozen      : age up to {int(frozen_age.max()):>2}, corr with y = {within_store_corr(frozen)}")

**The honest feature looks much worse, and it is supposed to.**

The naive lag validates beautifully and cannot be served: at serving time nobody
hands you yesterday's label for a day you are being asked to forecast. The gap
between those two numbers is **the size of the lie, not the size of the loss**.

0048 states this explicitly so it is not later mistaken for a bug and "fixed".

## 7. Why random folds are refused outright

Under forward chaining, "frozen at the end of the training fold" is a prefix in
time and age counts real elapsed days. Under random k-fold the training fold is
scattered *through* time, so a validation row's most recent stored observation is
whichever neighbour happened to land in the other fold.

In [ ]:
random_folds = make_folds(panel["sales"], "regression", n_splits=4)

try:
    fit_history(panel, ["store"], "sales", "date", folds=random_folds)
except ValueError as e:
    print("REFUSED:", e)

The number is computable. Here is what it would have been — the ages a validation
row would see under random folds, against the ages forward chaining produces:

In [ ]:
tr, va_rand = random_folds[0]
ages = []
for r in va_rand:
    same = [i for i in tr if panel["store"][i] == panel["store"][r]
            and panel["date"][i] < panel["date"][r]]
    if same:
        newest = max(same, key=lambda i: panel["date"][i])
        ages.append((panel["date"][r] - panel["date"][newest]).days)

print("random folds    :", pd.Series(ages).value_counts().sort_index().to_dict())
print("forward chaining:", frozen_age.iloc[s1_rows].dropna().astype(int).tolist())

That contrast is the argument. Random folds teach the model that **yesterday is
almost always available** — four rows in five sit at age 1. Forward chaining, and
real serving, look like `1, 2, 3, … 24`: age grows smoothly across the horizon.

Change the seed and both the value and its age change, because the holes punched
in history are an artifact of the RNG. That is a *different distribution*, not a
noisier one — which is why re-running with another seed does not converge on the
right answer, and why this is a refusal (0013) rather than a warning. A warning
that does not block is a warning that gets ignored.

## 8. Through the harness

Same machinery, driven by the tool the agent actually calls. The ordering column
is the one the run is split on; there has to be one, which is the same refusal
seen from the other side.

In [ ]:
import os
import tempfile

from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import finalize_run, fit_history as fit_history_tool, train_tabular_model
from tabular_agent.pipeline import load_bundle, predict_bundle
from tabular_agent.taskspec import TaskSpec

run_root = tempfile.mkdtemp()
src = os.path.join(run_root, "panel.csv")
panel.to_csv(src, index=False)

session = RunSession(
    os.path.join(run_root, "run"), src,
    TaskSpec(target="sales", task_type="regression", metric="rmse", n_splits=4),
    split_on="date",
)

out = fit_history_tool(session, by=["store"], value="sales")
print({k: out[k] for k in ("table", "new_columns", "n_groups", "n_rows_without_value")})
print()
print(out["note"])

Note the tool may point at **the target**, where `fit_group_stats` refuses it. A
per-group *statistic* of the target is target encoding and has its own tool; a
per-group *most recent* target is neither, and it cannot be smuggled out of a
served frame either — the target is not in one, so it can only ever come from
stored history.

Now train, finalize, and ask the bundle about days after everything it ever saw:

In [ ]:
train_tabular_model(session, table=out["table"])
fin = finalize_run(session)
bundle = load_bundle(os.path.join(session.run_dir, fin["bundle_dir"]))

future_rows = pd.DataFrame({
    "store": ["s1", "s2", "s1"],
    "date": pd.to_datetime(["2015-05-02", "2015-05-02", "2015-06-15"]),
    "promo": [0, 1, 0],
})

future_rows.assign(prediction=predict_bundle(bundle, future_rows).round(2))

The lineage replayed the stored history and answered rows dated weeks past the
last row it was fitted on — with no `sales` column in the request, because there
never is one.

---

# Slice 3 — a window, and what averaging is actually worth

Everything above reads **one observation** out of the stored tail. 0049 named the
problem with that in one sentence: *the most recent observation is a sample of
size one.* A window aggregates the tail instead of picking a point out of it.

But 0050 changed the argument for it, and the change matters. Rossmann v8 found
that the four best tables in the run were history tables **and so was the worst**,
separated by one parameter — and the reason was not sample size, it was the
**anchor**:

| | `_latest` | an offset like `364_ago` |
|---|---|---|
| anchored to | the **freeze boundary** — one date, shared by every served row | **the row** — a different date per row |
| so a bad boundary | is inherited by the whole batch | affects one row |

`_latest` took a working table from 0.200 to 0.464. The offsets *gained* 0.022.
Same tool, same key, same value.

So the window has to be **row-anchored** — `[row - w, row)` — or it is `_latest`
with extra arithmetic. And its honest claim is narrower than 0049 thought:
offsets already deliver recency *and* periodicity (364 days is 52 weeks, so the
weekday stays aligned). **A window adds only the averaging.** Whether that is
worth anything is a question with a real chance of answering "no", and
`scripts/probe_rossmann_windows.py` registers that possibility before measuring.

## 6. The window moves with the row

The contrast to watch below is between the two kinds of column on the same three
rows: `_latest` is one number repeated, the window is three different ones.

In [ ]:
from tabular_agent.encoders import HistoryEncoder

win = HistoryEncoder(
    ["store"], "sales", "date",
    windows=[(14, "mean"), (14, "std")], keep=60,
).fit(panel)

# Three rows marching away from the end of stored history.
ahead = pd.DataFrame({
    "store": ["s1"] * 3,
    "date": pd.to_datetime(["2015-05-02", "2015-05-09", "2015-05-16"]),
})

win.transform(ahead).set_index("date").round(2)

`store_sales_latest` is **the same number three times** — one draw from one date,
which is exactly the shape that cost v8 its worst table. The `14_mean` column
falls away smoothly instead, and `14_count` says why: the window keeps sliding
past the end of stored history, so it holds fewer and fewer observations until it
holds none.

**That count is not optional, and this is the reason.** A mean over fourteen
observations and a mean over one are the same number wearing the same name. Only
one of them is a summary.

In [ ]:
counts = win.transform(pd.DataFrame({
    "store": ["s1"] * 6,
    "date": pd.date_range("2015-05-01", periods=6, freq="3D"),
}))
counts.set_index("date")[["store_sales_latest", "store_sales_14_mean", "store_sales_14_count"]].round(2)

## 7. The row is never inside its own window

`_latest` relies on the per-fold freeze to keep a row out of its own feature
(0048). **A window must not**, and the difference is about what a failure would
look like: a self-included `_latest` shows up as a perfect correlation with the
target, while a self-included point in a mean over ninety is diluted to
invisibility. A guarantee that only holds when someone checks the numbers is not
a guarantee, so the right endpoint is strict — `[row - w, row)` stops *before* the
row's own ordering value.

Below, the encoder is fitted on the panel and then asked about **the same rows**,
which is the situation the freeze would normally rule out.

In [ ]:
self_check = win.transform(panel)
s1 = self_check[self_check["store"] == "s1"].head(6)

s1[["date", "sales", "store_sales_14_mean", "store_sales_14_count"]].set_index("date").round(2)

The first row has a count of **0**, not 1: it has no past, and it is not its own
past. Each later row's mean is over the rows strictly above it.

Note also what an empty window is *not*. A count of 0 means "this group had
nothing in that stretch"; a group never seen at all gets `NaN` even for the count,
because nothing is known about it. Those are different states and they read
differently.

In [ ]:
never_seen = win.transform(pd.DataFrame({"store": ["s99"], "date": [pd.Timestamp("2015-05-02")]}))
print("unseen group  :", never_seen[["store_sales_14_mean", "store_sales_14_count"]].to_dict("records"))

way_past = win.transform(pd.DataFrame({"store": ["s1"], "date": [pd.Timestamp("2016-01-01")]}))
print("empty window  :", way_past[["store_sales_14_mean", "store_sales_14_count"]].to_dict("records"))

## 8. What the menu refuses, and why

Three refusals, each with a reason that is not about this dataset.

**A window shorter than the forecast horizon.** Stored history stops one horizon
before the earliest row you predict, so a window that short covers only the gap
and is empty for *every* served row. An offset that short degrades into
`_latest`; a window that short is nothing at all.

In [ ]:
try:
    fit_history(panel, ["store"], "sales", "date", folds=folds,
                windows=[(3, "mean")], forecast_horizon=10)
except ValueError as e:
    print("REFUSED:", e)

**`min`, `max` and `sum`.** This is 0046's reasoning at a shorter range. An
extreme over a trailing window is close to a single observation, which is the
thing 0049 was written about; `sum` is `count * mean` and the count is emitted
beside every window already, so it is a column a model can form for itself.

In [ ]:
from tabular_agent.encoders import WINDOW_STATS

print("on the menu:", WINDOW_STATS)
try:
    HistoryEncoder(["store"], "sales", "date", windows=[(30, "max")])
except ValueError as e:
    print("REFUSED:", e)

**A window too long to store.** `keep` is capped at 1000 observations per group,
which is what keeps an encoder's size independent of the table's length. A window
that needs more is **refused rather than quietly served from a shortened tail** — a
365-day window that silently averages whatever fits is a wrong number under a
confident name, which is the failure this whole area keeps producing.

The escape hatch is the caller's knowledge, not a guess: a series with weekly
readings needs 52 slots for a year, not 365, and only its owner knows that.

In [ ]:
try:
    HistoryEncoder(["store"], "sales", "date", windows=[(5000, "mean")])
except ValueError as e:
    print("REFUSED:", e)

sparse = HistoryEncoder(["store"], "sales", "date", windows=[(5000, "mean")], keep=60)
print("\naccepted with an explicit keep:", sparse.keep)

## 9. Through the harness, and the number to read first

The tool reports `window_observations` — the median number of observations each
window actually held — for the same reason it reports `offsets_answerable`. On
Rossmann v8 a 7-day offset over a 44-day block was honoured for **11%** of rows and
nothing said so until someone went looking. A window that averages two points is
that same failure in new clothing, and this is the line that says so.

In [ ]:
# The session above was finalized, and a packaged run will not be built on again.
# A window is a fresh question, so it gets a fresh run.
session_w = RunSession(
    os.path.join(run_root, "run_windows"), src,
    TaskSpec(target="sales", task_type="regression", metric="rmse", n_splits=4,
             forecast_horizon=1.0),
    split_on="date",
)

out_w = fit_history_tool(
    session_w, by=["store"], value="sales",
    windows=[[21, "mean"], [21, "median"]], keep=60,
)

print({k: out_w[k] for k in ("new_columns", "window_observations", "block_span")})
print()
print(out_w["note"])


`block_span` is how wide a scored block is, in the ordering column's units. It is
reported because **the agent has no other way to learn it** — `inspect_data` gives
`n_val` as a row count and nothing converts that to a span — and "use a window
longer than the period you predict" is useless advice without it.

## 10. And then the measurement said no

Everything above is the capability working. Whether it is *worth* anything is a
separate question, and on Rossmann the answer came back negative. Full detail in
0052; the two tables that matter:

**The key is the gain, not the averaging.** Same tool, same 364 days, same
statistic — only the group key differs:

| key | probe | vs baseline |
|---|---|---|
| `Store` | 0.22072 | −0.00263 |
| `Store` + `DayOfWeek` | 0.17352 | **+0.04458** |

**And the trailing bound — the thing slice 3 exists to add — loses to the
unbounded mean of the same key**, which 0046 already built:

| | probe | vs baseline |
|---|---|---|
| all-time target mean of `Store_DOW` (0046) | **0.15762** | **+0.06048** |
| trailing 364 mean of `Store`+`DayOfWeek` (slice 3) | 0.17352 | +0.04458 |
| both | 0.16944 | +0.04866 |

So the competition winner's *"sales for this store on a typical Monday over the
last six months"* is two halves, and **the half this project was missing was the
periodic key** — already built, already paid for. `docs/slice3_windows.md` said in
advance that this was a publishable result and not something to argue away.

## 11. The failure the count could not see

Sweeping the length exposed something the design did not anticipate. The
prediction registered beforehand was *monotone improvement with length*, and it
was wrong:

| window | probe | observations |
|---|---|---|
| 91 | 0.23365 | 3 |
| 364 | **0.17352** | 39 |
| 546 | 0.21840 | 58 |
| 728 | 0.31097 | 59 |

546 and 728 hold **the same number of observations** and are 0.093 apart, so
coverage is not the mechanism. This is:

> **A window longer than the shortest fold's training prefix cannot be filled in
> training, while a served row fills it completely.**

Forward chaining trains fold 0 on a short head of the data — 136 days out of 850
here. So a 728-day window trained at a median of 59 observations and was *served*
at 101. The model learns "a mean over ~59" and is handed "a mean over ~101": one
name, two quantities. That is train/serve skew arriving through a door 0048 did
not cover — not *when* history stops, but *how much* of it there was.

**And `window_observations`, added the same day to catch exactly this kind of
mistake, rated the broken window higher than the good one** (59 against 39),
because the damage lives in the ratio between the training and serving sides and
the count only ever sees one side. So the tool now reports
`shortest_training_prefix` too, and says outright that the count is no guide here.


In [ ]:
out_long = fit_history_tool(
    session_w, by=["store"], value="sales",
    windows=[[100, "mean"]], keep=200,
)

print("shortest training prefix:", out_long["shortest_training_prefix"])
print("window held (median)    :", out_long["window_observations"])
print()
print(out_long["note"].split("WARNING:")[-1].strip())

## What is left

- **Slice 3 shipped and lost.** The capability is general, correct and cheap, and
  one dataset is one reading — Rossmann is unusually stable per store-weekday, so
  the feature was tested where it had least to offer. 0052 leads with the loss so
  a later session does not rediscover windows as an obvious win.
- **`median` is kept but unvalidated.** Its argument (a median survives the spike
  0050 documented) was not refuted, only unsupported: 0.0003 from the mean on the
  good key, clearly worse on `Store`.
- **None of these numbers has a yardstick.** Under an ordered split validation is
  one period and `val_se` is `None` (0022, 0036, 0041). The directions are
  consistent across three comparisons; the gaps have no error bars. The paired
  bootstrap of 0036 is still not wired in, and it is what would fix this.
- **`_latest` is still emitted unconditionally**, so every history table carries
  the pair that destroyed v8's t7.
- **The time unit.** Dates always become days, so a second-resolution log gets an
  age of `0.00347`. The fix is a *declared* unit (0051), not built.
- **Known future covariates** and **recursive forecasting** remain where 0047 and
  0048 left them.